# Gikomba Stall Analysis: How Is Wanjiku's Business Doing?

**Analyst:** Chala Wakshuma  |  **Cohort:** Chala Wakshuma

Wanjiku runs a fresh-produce stall at Gikomba Market. She has two weeks of daily sales and a rough stock record, and wants to know: *How is my business doing? Is it growing? What should I restock, and where should I focus?*

This notebook uses **core Python only** (no pandas, no NumPy) to answer those questions in seven tasks.

## Dataset

The data below is copied exactly as provided: 14 days of sales (KES), the matching day names, and the stock record for five products.

In [ ]:
daily_sales = [2300, 1950, 2800, 2100, 3600, 5200, 4100,
               2500, 2200, 3100, 2400, 3900, 5600, 4700]   # 14 days, KES
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] * 2

# qty = kg in stock, reorder = reorder level (kg), cost/price = KES per kg
stock = {
    "Tomatoes": {"qty": 18, "reorder": 25, "cost": 60, "price": 90},
    "Onions":   {"qty": 40, "reorder": 20, "cost": 70, "price": 100},
    "Sukuma":   {"qty": 9,  "reorder": 15, "cost": 30, "price": 50},
    "Potatoes": {"qty": 55, "reorder": 30, "cost": 50, "price": 75},
    "Avocado":  {"qty": 12, "reorder": 10, "cost": 80, "price": 140},
}

## Task 1: Total, average, highest and lowest daily sales

**Approach:** Use `sum()`, `len()`, `max()` and `min()` on the 14 values. Then use `list.index()` to find the position of the best and worst day. The position gives the day name (via the `days` list) and the week number (positions 0 to 6 are week 1, positions 7 to 13 are week 2, so `position // 7 + 1`).

In [ ]:
total_sales = sum(daily_sales)
average_sales = total_sales / len(daily_sales)
highest_sales = max(daily_sales)
lowest_sales = min(daily_sales)

best_pos = daily_sales.index(highest_sales)
worst_pos = daily_sales.index(lowest_sales)

best_day, best_week = days[best_pos], best_pos // 7 + 1
worst_day, worst_week = days[worst_pos], worst_pos // 7 + 1

print(f"Total sales   : KES {total_sales:,}")
print(f"Average daily : KES {average_sales:,.2f}")
print(f"Highest day   : KES {highest_sales:,} ({best_day}, week {best_week})")
print(f"Lowest day    : KES {lowest_sales:,} ({worst_day}, week {worst_week})")

Total sales   : KES 46,450
Average daily : KES 3,317.86
Highest day   : KES 5,600 (Sat, week 2)
Lowest day    : KES 1,950 (Tue, week 1)


## Task 2: Weekly totals and week-on-week growth percentage

**Approach:** `week_totals()` slices the first seven and the last seven values, sums each, and returns them as a tuple. I unpack the tuple into two variables and compute growth as (week 2 − week 1) ÷ week 1 × 100.

In [ ]:
def week_totals(sales):
    """Return a tuple (week 1 total, week 2 total) from a list of 14 daily sales.

    Week 1 is the slice sales[0:7] and week 2 is the slice sales[7:14].
    """
    week1 = sum(sales[:7])
    week2 = sum(sales[7:14])
    return (week1, week2)


week1_total, week2_total = week_totals(daily_sales)
growth_pct = (week2_total - week1_total) / week1_total * 100

print(f"Week 1 total : KES {week1_total:,}")
print(f"Week 2 total : KES {week2_total:,}")
print(f"Week-on-week growth: {growth_pct:.2f}%")

Week 1 total : KES 22,050
Week 2 total : KES 24,400
Week-on-week growth: 10.66%


## Task 3: Classifying each day as Strong, Normal or Weak

**Approach:** `classify_day()` compares a day's sales with the 14-day average. More than 20% above the average (above `average * 1.2`) is **Strong**, more than 20% below (below `average * 0.8`) is **Weak**, and everything else is **Normal**. I apply it to all 14 days and count the categories with the dictionary counting pattern (`counts[label] = counts.get(label, 0) + 1`).

In [ ]:
def classify_day(amount, average):
    """Classify a day's sales relative to the average.

    Returns "Strong" if amount is more than 20% above average,
    "Weak" if it is more than 20% below average, otherwise "Normal".
    """
    if amount > average * 1.2:
        return "Strong"
    elif amount < average * 0.8:
        return "Weak"
    else:
        return "Normal"


day_labels = []
day_counts = {}
for day, amount in zip(days, daily_sales):
    label = classify_day(amount, average_sales)
    day_labels.append(label)
    day_counts[label] = day_counts.get(label, 0) + 1

for i in range(len(daily_sales)):
    print(f"Week {i // 7 + 1} {days[i]}: KES {daily_sales[i]:>5,} -> {day_labels[i]}")

print()
print("Counts:", day_counts)

Week 1 Mon: KES 2,300 -> Weak
Week 1 Tue: KES 1,950 -> Weak
Week 1 Wed: KES 2,800 -> Normal
Week 1 Thu: KES 2,100 -> Weak
Week 1 Fri: KES 3,600 -> Normal
Week 1 Sat: KES 5,200 -> Strong
Week 1 Sun: KES 4,100 -> Strong
Week 2 Mon: KES 2,500 -> Weak
Week 2 Tue: KES 2,200 -> Weak
Week 2 Wed: KES 3,100 -> Normal
Week 2 Thu: KES 2,400 -> Weak
Week 2 Fri: KES 3,900 -> Normal
Week 2 Sat: KES 5,600 -> Strong
Week 2 Sun: KES 4,700 -> Strong

Counts: {'Weak': 6, 'Normal': 4, 'Strong': 4}


## Task 4: Weekdays vs Weekends

**Approach:** Using the `days` list, I collect sales on Monday to Friday into one list and Saturday and Sunday into another across both weeks, take the average of each, and express the weekend premium as (weekend average − weekday average) ÷ weekday average × 100.

In [ ]:
weekend_names = ["Sat", "Sun"]
weekday_sales = []
weekend_sales = []

for day, amount in zip(days, daily_sales):
    if day in weekend_names:
        weekend_sales.append(amount)
    else:
        weekday_sales.append(amount)

weekday_avg = sum(weekday_sales) / len(weekday_sales)
weekend_avg = sum(weekend_sales) / len(weekend_sales)
weekend_premium_pct = (weekend_avg - weekday_avg) / weekday_avg * 100

print(f"Weekday average (Mon-Fri): KES {weekday_avg:,.2f}  ({len(weekday_sales)} days)")
print(f"Weekend average (Sat-Sun): KES {weekend_avg:,.2f}  ({len(weekend_sales)} days)")
print(f"Weekends sell {weekend_premium_pct:.1f}% more per day than weekdays")

Weekday average (Mon-Fri): KES 2,685.00  (10 days)
Weekend average (Sat-Sun): KES 4,900.00  (4 days)
Weekends sell 82.5% more per day than weekdays


## Task 5: Restock list of Products and Gross Margins

**Approach:** `products_to_reorder()` loops through `stock.items()` and collects every product whose `qty` is at or below its `reorder` level. A second loop computes gross margin % as (price − cost) ÷ price × 100 for each product.

In [ ]:
def products_to_reorder(stock_data):
    """Return a list of product names whose quantity is at or below the reorder level."""
    low = []
    for name, info in stock_data.items():
        if info["qty"] <= info["reorder"]:
            low.append(name)
    return low


reorder_list = products_to_reorder(stock)
print("Products to reorder:", reorder_list)
print()

margins = {}
for name, info in stock.items():
    margins[name] = (info["price"] - info["cost"]) / info["price"] * 100

print(f"{'Product':<10} {'Cost':>6} {'Price':>6} {'Margin':>8}")
for name, info in stock.items():
    print(f"{name:<10} {info['cost']:>6} {info['price']:>6} {margins[name]:>7.1f}%")

best_margin_product = max(margins, key=margins.get)
worst_margin_product = min(margins, key=margins.get)
print()
print(f"Highest margin: {best_margin_product} ({margins[best_margin_product]:.1f}%)")
print(f"Lowest margin : {worst_margin_product} ({margins[worst_margin_product]:.1f}%)")

Products to reorder: ['Tomatoes', 'Sukuma']

Product      Cost  Price   Margin
Tomatoes       60     90    33.3%
Onions         70    100    30.0%
Sukuma         30     50    40.0%
Potatoes       50     75    33.3%
Avocado        80    140    42.9%

Highest margin: Avocado (42.9%)
Lowest margin : Onions (30.0%)


## Task 6: Stall Performance Report

**Approach:** Pull every result from the earlier tasks into one report printed with f-strings. Money uses the `:,` format specifier so KES amounts get thousands separators.

In [ ]:
line = "=" * 46
print(line)
print("        STALL PERFORMANCE REPORT")
print("        Wanjiku's Stall, Gikomba Market")
print(line)

print("\nSALES SUMMARY (14 days)")
print(f"  Total sales        : KES {total_sales:,}")
print(f"  Average daily      : KES {average_sales:,.2f}")
print(f"  Best day           : {best_day}, week {best_week} (KES {highest_sales:,})")
print(f"  Worst day          : {worst_day}, week {worst_week} (KES {lowest_sales:,})")

print("\nGROWTH")
print(f"  Week 1 total       : KES {week1_total:,}")
print(f"  Week 2 total       : KES {week2_total:,}")
print(f"  Week-on-week       : {growth_pct:+.2f}%")

print("\nDAY QUALITY")
for label in ["Strong", "Normal", "Weak"]:
    print(f"  {label:<7}: {day_counts.get(label, 0)} days")

print("\nWEEKDAY vs WEEKEND")
print(f"  Weekday average    : KES {weekday_avg:,.2f}")
print(f"  Weekend average    : KES {weekend_avg:,.2f}")
print(f"  Weekend premium    : {weekend_premium_pct:+.1f}%")

print("\nSTOCK")
print(f"  Reorder now        : {', '.join(reorder_list)}")
print("  Gross margins      :")
for name in margins:
    print(f"    {name:<10} {margins[name]:.1f}%")
print(line)

        STALL PERFORMANCE REPORT
        Wanjiku's Stall, Gikomba Market

SALES SUMMARY (14 days)
  Total sales        : KES 46,450
  Average daily      : KES 3,317.86
  Best day           : Sat, week 2 (KES 5,600)
  Worst day          : Tue, week 1 (KES 1,950)

GROWTH
  Week 1 total       : KES 22,050
  Week 2 total       : KES 24,400
  Week-on-week       : +10.66%

DAY QUALITY
  Strong : 4 days
  Normal : 4 days
  Weak   : 6 days

WEEKDAY vs WEEKEND
  Weekday average    : KES 2,685.00
  Weekend average    : KES 4,900.00
  Weekend premium    : +82.5%

STOCK
  Reorder now        : Tomatoes, Sukuma
  Gross margins      :
    Tomatoes   33.3%
    Onions     30.0%
    Sukuma     40.0%
    Potatoes   33.3%
    Avocado    42.9%



## Task 7: Recommendations for Wanjiku

** Each recommendation below is tied to a specific number produced in Tasks 1 to 6.

1. **Your business is growing.** Week 2 sales were KES 24,400 against KES 22,050 in week 1, a 10.66% rise, so keep doing what you did in week 2.
2. **Focus your best stock and your longest hours on Saturday and Sunday.** Weekends averaged KES 4,900 a day against KES 2,685 on weekdays (+82.5%), and those 4 weekend days brought in KES 19,600, about 42% of all sales.
3. **Restock Tomatoes and Sukuma now.** Tomatoes are at 18 kg against a reorder level of 25 kg, and Sukuma is at 9 kg against 15 kg, so both are below the line; the other three products are still above it.
4. **Push Avocado and Sukuma, and watch Onions.** Avocado earns the highest margin at 42.9% and Sukuma follows at 40.0%, while Onions earns only 30.0%, so give the better-margin items the best spot on the stall.
5. **Use the quiet days to lift sales.** Six of the 14 days were "Weak" (all Mondays, Tuesdays and Thursdays), with the lowest day at KES 1,950, so try small weekday offers or buy less stock for those days to reduce waste.
